# معالجة البيانات وبناء خط المعالجة (Unsupervised Preprocessing Pipeline)

## مقدمة
بناءً على ملاحظة أن فئة الاحتيال (Class 1) تشكل أقل من 1% من البيانات، سننتقل إلى نهج **التعلم غير الأشرافي (Unsupervised Learning)**. في هذا النهج، سنقوم بتدريب النموذج على البيانات السليمة فقط (Class 0) ليتعلم النمط الطبيعي للعمليات، ومن ثم نعتبر أي انحراف عن هذا النمط بمثابة احتيال (Anomaly).

## ماذا سنفعل في هذا الـ Notebook؟
1. تحميل مجموعة البيانات.
2. فصل البيانات السليمة (Class 0) عن بيانات الاحتيال (Class 1).
3. تقسيم البيانات السليمة إلى مجموعات تدريب واختبار.
4. دمج بيانات الاحتيال بالكامل في مجموعة الاختبار لتقييم قدرة النموذج على اكتشافها.
5. بناء خط معالجة (Pipeline) لتوحيد مقاييس البيانات (Scaling).

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib
import os

# تحميل البيانات
df = pd.read_csv("../data/creditcard.csv")
print(f"إجمالي عدد السجلات: {len(df)}")
print(f"توزيع الفئات:\n{df['Class'].value_counts(normalize=True) * 100}")

إجمالي عدد السجلات: 284807
توزيع الفئات:
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


## 2. تجهيز البيانات للتعلم غير الأشرافي
سنقوم بعزل الكلاس 0 للتدريب.

In [14]:
# فصل البيانات السليمة عن الاحتيال
normal_data = df[df['Class'] == 0]
fraud_data = df[df['Class'] == 1]

print(f"عدد العمليات السليمة: {len(normal_data)}")
print(f"عدد عمليات الاحتيال: {len(fraud_data)}")

# تقسيم العمليات السليمة فقط إلى تدريب واختبار
# سنستخدم 80% من البيانات السليمة للتدريب
X_train_normal, X_test_normal = train_test_split(
    normal_data.drop('Class', axis=1), 
    test_size=0.2, 
    random_state=42
)

# مجموعة الاختبار ستحتوي على الـ 20% المتبقية من السليمة + كل عمليات الاحتيال
X_test = pd.concat([X_test_normal, fraud_data.drop('Class', axis=1)])
y_test = np.concatenate([
    np.zeros(len(X_test_normal)), 
    np.ones(len(fraud_data))
])

print(f"حجم بيانات التدريب (سليمة فقط): {X_train_normal.shape}")
print(f"حجم بيانات الاختبار (سليمة + احتيال): {X_test.shape}")

عدد العمليات السليمة: 284315
عدد عمليات الاحتيال: 492
حجم بيانات التدريب (سليمة فقط): (227452, 30)
حجم بيانات الاختبار (سليمة + احتيال): (57355, 30)


## 3. بناء الـ Pipeline
سنستخدم `StandardScaler` لتوحيد المقاييس، وهو أمر ضروري لمعظم خوارزميات اكتشاف الشذوذ.

In [15]:
pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# ملائمة الـ Pipeline على بيانات التدريب السليمة فقط
X_train_scaled = pipeline.fit_transform(X_train_normal)
X_test_scaled = pipeline.transform(X_test)

print("تمت معالجة البيانات وتجهيز الـ Pipeline بنجاح.")

تمت معالجة البيانات وتجهيز الـ Pipeline بنجاح.


## 4. حفظ البيانات المعالجة
سنقوم بحفظ البيانات لاستخدامها في ملف التدريب.

In [16]:
# ملاحظة: في بيئة العمل الحقيقية نستخدم joblib أو نمرر البيانات مباشرة
print("البيانات جاهزة للمرحلة القادمة: تدريب نماذج Unsupervised.")

البيانات جاهزة للمرحلة القادمة: تدريب نماذج Unsupervised.
